In [2]:
import pandas as pd 
import ta

df = pd.read_csv("../data/AAPL.csv")
df.columns = [c.lower() for c in df.columns]

df.head()


,price,close,high,low,open,volume
0,Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
1,Date,NaN,NaN,NaN,NaN,NaN
2,2015-12-21,24.1995849609375,24.208603926159757,23.80275909211675,24.188310824361743,190362400
3,2015-12-22,24.17703628540039,24.287515589338216,24.00116905929387,24.215365594086148,131157600
4,2015-12-23,24.488176345825195,24.542288397916,24.17026438438559,24.186047137920035,130629600


In [3]:
import pandas as pd
import ta

df = pd.read_csv(
    "../data/AAPL.csv",
    skiprows=[1, 2]
)

df = df.rename(
    columns={
        "Price": "date",
        "Close": "close",
        "High": "high",
        "Low": "low",
        "Open": "open",
        "Volume": "volume",
    }
)

df.columns = df.columns.str.lower()

df["date"] = pd.to_datetime(
    df["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

numeric_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume",
]

df[numeric_columns] = df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

df = (
    df.dropna(subset=["date", *numeric_columns])
      .sort_values("date")
      .reset_index(drop=True)
)

df = df[
    [
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]
]

df.head()

,date,open,high,low,close,volume
0,2015-12-21,24.188311,24.208604,23.802759,24.199585,190362400
1,2015-12-22,24.215366,24.287516,24.001169,24.177036,131157600
2,2015-12-23,24.186047,24.542288,24.170264,24.488176,130629600
3,2015-12-24,24.576112,24.576112,24.339369,24.357407,54281600
4,2015-12-28,24.258203,24.280751,23.940293,24.084593,106816800


In [4]:
print(df.dtypes)
print(df.isna().sum())
print(df.shape)
print(df["date"].min(), df["date"].max())

date      datetime64[us]
open             float64
high             float64
low              float64
close            float64
volume             int64
dtype: object
date      0
open      0
high      0
low       0
close     0
volume    0
dtype: int64
(2515, 6)
2015-12-21 00:00:00 2025-12-19 00:00:00


In [5]:
df["rsi_7"] = ta.momentum.RSIIndicator(
    close=df["close"],
    window=7
).rsi()

df["rsi_14"] = ta.momentum.RSIIndicator(
    close=df["close"],
    window=14
).rsi()

In [6]:
df["cci_7"] = ta.trend.CCIIndicator(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=7
).cci()

df["cci_14"] = ta.trend.CCIIndicator(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=14
).cci()

In [7]:
df["sma_50"] = ta.trend.SMAIndicator(
    close=df["close"],
    window=50
).sma_indicator()

df["ema_50"] = ta.trend.EMAIndicator(
    close=df["close"],
    window=50
).ema_indicator()

df["sma_100"] = ta.trend.SMAIndicator(
    close=df["close"],
    window=100
).sma_indicator()

df["ema_100"] = ta.trend.EMAIndicator(
    close=df["close"],
    window=100
).ema_indicator()

In [8]:
macd = ta.trend.MACD(close=df["close"])

df["macd"] = macd.macd()

In [9]:
bb = ta.volatility.BollingerBands(
    close=df["close"],
    window=20,
    window_dev=2
)

df["bollinger"] = bb.bollinger_hband()

In [10]:
previous_close = df["close"].shift(1)

true_range_components = pd.concat(
    [
        df["high"] - df["low"],
        (df["high"] - previous_close).abs(),
        (df["low"] - previous_close).abs(),
    ],
    axis=1,
)

df["TrueRange"] = true_range_components.max(axis=1)

In [11]:
df["atr_7"] = ta.volatility.AverageTrueRange(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=7
).average_true_range()

df["atr_14"] = ta.volatility.AverageTrueRange(
    high=df["high"],
    low=df["low"],
    close=df["close"],
    window=14
).average_true_range()

In [12]:
df["next_day_close"] = df["close"].shift(-1)

In [13]:
df = df.dropna().reset_index(drop=True)

In [14]:
df = df[
    [
        "date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "rsi_7",
        "rsi_14",
        "cci_7",
        "cci_14",
        "sma_50",
        "ema_50",
        "sma_100",
        "ema_100",
        "macd",
        "bollinger",
        "TrueRange",
        "atr_7",
        "atr_14",
        "next_day_close",
    ]
]

In [15]:
df.to_csv("AAPL_engineered.csv", index=False)

In [16]:
print(df.head())
print(df.columns)
print(df.shape)

        date       open       high        low      close     volume  \
0 2016-05-13  20.526531  20.907412  20.526531  20.645128  177571200   
1 2016-05-16  21.071627  21.527773  20.902854  21.411455  245039200   
2 2016-05-17  21.564266  21.598476  21.213034  21.322508  187667600   
3 2016-05-18  21.475312  21.714787  21.413732  21.566540  168249600   
4 2016-05-19  21.584794  21.584794  21.340757  21.484442  121768400   

       rsi_7     rsi_14       cci_7      cci_14     sma_50     ema_50  \
0  18.165817  23.979977 -121.638643 -110.854532  23.417778  22.794850   
1  47.605299  38.529057   70.931992    1.472171  23.378976  22.740599   
2  45.393943  37.628871   83.412084   57.388288  23.343563  22.684988   
3  52.462071  41.656372  110.755574  136.677076  23.316839  22.641127   
4  49.925657  40.704101   69.665180   84.663171  23.288065  22.595767   

     sma_100    ema_100      macd  bollinger  TrueRange     atr_7    atr_14  \
0  22.886907  23.079897 -0.857738  24.981912   0.380881

In [17]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
date,2415,2021-02-28 14:03:43.602484,2016-05-13 00:00:00,2018-10-04 12:00:00,2021-03-02 00:00:00,2023-07-25 12:00:00,2025-12-18 00:00:00,NaN
open,2415.0,117.101104,20.526531,44.154255,124.228556,172.36299,286.200012,71.994981
high,2415.0,118.379921,20.907412,44.45396,125.293044,174.50753,288.619995,72.755755
low,2415.0,115.92925,20.526531,43.830798,122.988338,171.08711,283.299988,71.300369
close,2415.0,117.214057,20.645128,44.159698,124.121147,172.270706,286.190002,72.06539
volume,2415.0,99026646.625259,20135600.0,60772350.0,87222800.0,119778200.0,447940000.0,54943325.352683
rsi_7,2415.0,56.343275,10.002496,42.633297,57.755884,70.593477,94.513055,17.685124
rsi_14,2415.0,56.002805,20.057878,46.16418,56.919051,65.29752,90.695842,12.860711
cci_7,2415.0,20.376791,-233.333333,-65.63804,41.019935,99.717565,233.333333,100.078389
cci_14,2415.0,28.429906,-359.820844,-58.364292,52.691541,110.385407,343.350307,108.544371
